# 🐾 Animal Sound Generator — v15 Latent Diffusion

**Industry standard.** Frozen encoder → latent diffusion → decoder → Griffin-Lim.

| Phase | Script | Time (L4) |
|-------|--------|:---------:|
| 1 | Train Decoder | ~20 min |
| 2 | Train Latent Diffusion | ~15 min |
| 3 | Generate & Download | ~2 min |

### Why v15 (v1-v14 all failed)
- **Diffuses on 2,240 latent values** (not 35,328 mel bins)
- **Decoder without skip connections** (self-contained generator)
- **Frozen 149M encoder** (proven MSE=0.015)
- **Griffin-Lim audio** (no HiFi-GAN electric noise)

### Before running:
1. Runtime → L4 GPU
2. Push code to GitHub first

In [ ]:
# @title 1. Setup
!git clone https://github.com/weseegod/animal_sound_generator.git /content/animal_sound_generator
%cd /content/animal_sound_generator
!git pull

!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy matplotlib tqdm soundfile

!mkdir -p models

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# @title 2. Download ESC-50 Data (640 clean animal sound clips)
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /tmp/esc50.zip
!unzip -qo /tmp/esc50.zip -d /tmp/
!python src/scripts/setup_esc50.py --source /tmp/ESC-50-master/audio --target data/esc50

!ls data/esc50/

# Restore encoder checkpoint from Drive (568MB - too big for GitHub)
import os
DRIVE = "/content/drive/MyDrive/animal_sound_generator/models"
from google.colab import drive
drive.mount("/content/drive")
if os.path.isdir(DRIVE):
    !cp {DRIVE}/best_autoencoder_train.pth models/ 2>/dev/null
    if os.path.exists('models/best_autoencoder_train.pth'):
        print('✅ Encoder checkpoint restored from Drive')
    else:
        print('⚠️  Upload best_autoencoder_train.pth to Drive first!')

In [ ]:
# @title 3. Phase 1: Train Decoder (~20 min)
!python src/latent_diff/train_decoder.py

In [ ]:
# @title 4. Phase 2: Train Latent Diffusion (~15 min)
!python src/latent_diff/train_diff.py

In [ ]:
# @title 5. Generate all 7 animal sounds
!python src/latent_diff/generate.py

import os
for f in sorted(os.listdir('outputs/generated')):
    if f.endswith('.wav'):
        size = os.path.getsize(f'outputs/generated/{f}') / 1024
        print(f'  {f} ({size:.0f} KB)')

In [ ]:
# @title 6. Save checkpoints + Download audio
!mkdir -p /content/drive/MyDrive/animal_sound_generator/models 2>/dev/null
!cp models/latent_decoder_best.pth models/latent_diffusion_best.pth /content/drive/MyDrive/animal_sound_generator/models/ 2>/dev/null

import zipfile, os
with zipfile.ZipFile('v15_animal_sounds.zip', 'w') as z:
    for f in os.listdir('outputs/generated'):
        if f.endswith('.wav'): z.write(f'outputs/generated/{f}', f)
from google.colab import files
files.download('v15_animal_sounds.zip')